PROYECTO 1 — LIMPIEZA DE DATOS
ESTABLECIMIENTOS EDUCATIVOS HASTA NIVEL DIVERSIFICADO — 22 DEPARTAMENTOS DE GUATEMALA

Curso: CC3084 – Data Science · Semestre II 2026 · Universidad del Valle de Guatemala

Fuente de los datos: http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/ (NIVEL ESCOLAR: DIVERSIFICADO)

Fecha de extracción: julio 2026

Este notebook cubre las actividades 5 a 9 de la guía: limpieza de todas las variables, registro de
transformaciones, validación automática, informe de calidad antes/después y generación del conjunto
limpio único. La lógica reproducible vive en code/limpieza_lib.py. Todo usa rutas relativas a la
raíz del repositorio, así que es reproducible en cualquier máquina que clone el repo.

Decisiones de diseño acordadas con el equipo:
1. Los 22 archivos vienen en dos estructuras (Grupo A = 18 col, Grupo B = 16 col). Se unifican a un
   solo esquema.
2. En el Grupo B, SUPERVISOR y DIRECTOR vienen fusionados. Para uniformar, en el Grupo A también
   se fusionan en SUPERVISOR_DIRECTOR.
3. El Grupo B tiene columnas desplazadas hacia DEPARTAMENTAL; se reparan parseando por dominios.


1. PREPARACIÓN DEL ENTORNO

In [1]:
import os, sys, glob, re
import pandas as pd
import numpy as np
from rapidfuzz import fuzz

# Ubicarse en la raíz del repo (este notebook está en code/).
if os.path.basename(os.getcwd()) == "code":
    os.chdir("..")
sys.path.insert(0, "code")
import limpieza_lib as L

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Directorio de trabajo:", os.getcwd())
print("Archivos en Data/:", len(glob.glob("Data/*.csv")))

Directorio de trabajo: C:\Users\mjyee\OneDrive\Documentos\GitHub\Proyecto1-DataScience
Archivos en Data/: 22


REGISTRO DE TRANSFORMACIONES
Acumulamos cada operación en una lista para producir, al final, la tabla que pide la actividad 6
(Variable · Problema detectado · Transformación · Registros afectados · Justificación).

In [2]:
REGISTRO = []
def log(variable, problema, transformacion, afectados, justificacion, es_correccion=True):
    # es_correccion=False para cambios estructurales (fusiones, drops) que no son
    # "errores corregidos" y no deben inflar ese conteo del informe.
    REGISTRO.append({
        "Variable": variable, "Problema detectado": problema,
        "Transformación": transformacion, "Registros afectados": afectados,
        "Justificación": justificacion, "_es_correccion": es_correccion})
    print(f"[LOG] {variable}: {transformacion} (afectados={afectados})")

2. CARGA DE LOS DATOS CRUDOS (22 ARCHIVOS, 2 GRUPOS)

In [3]:
A_raw, files_A = L.cargar_grupo_A()
B_raw, files_B = L.cargar_grupo_B()
print("Grupo A:", A_raw.shape, "->", len(files_A), "archivos")
print("Grupo B:", B_raw.shape, "->", len(files_B), "archivos")
print("Registros crudos totales:", len(A_raw) + len(B_raw))

Grupo A: (4033, 19) -> 11 archivos
Grupo B: (5673, 17) -> 11 archivos
Registros crudos totales: 9706


2.1 MÉTRICAS DEL ESTADO ANTES
Guardamos las métricas del conjunto crudo para el informe comparativo de la actividad 8. Se calculan
sobre la unión "tal cual" (concatenando ambos grupos por nombre de columna; las columnas que solo
existen en un grupo quedan como NA en el otro), que es el punto de partida real antes de limpiar.

In [4]:
crudo = pd.concat([A_raw, B_raw], ignore_index=True)
cols_crudo = [c for c in crudo.columns if c != "__ARCHIVO__"]

def metricas(df, cols):
    n = len(df); celdas = n * len(cols)
    na = int(df[cols].isna().sum().sum())
    return {
        "Registros": n,
        "Variables": len(cols),
        "Valores faltantes": f"{na} ({round(100*na/celdas,2)}%)",
        "Variables con NA": int((df[cols].isna().sum() > 0).sum()),
        "Duplicados exactos": int(df[cols].duplicated().sum()),
    }

ANTES = metricas(crudo, cols_crudo)
ANTES

{'Registros': 9706,
 'Variables': 19,
 'Valores faltantes': '29446 (15.97%)',
 'Variables con NA': 13,
 'Duplicados exactos': 0}

3. REPARACIÓN DEL DESPLAZAMIENTO DE COLUMNAS (GRUPO B)

En 817 registros del Grupo B, los valores de STATUS, MODALIDAD, JORNADA y PLAN quedaron
concatenados dentro de DEPARTAMENTAL y sus columnas propias quedaron vacías. Verificación previa:
los 817 registros con DEPARTAMENTAL contaminado son exactamente los mismos que tienen PLAN vacío.

L.reparar_departamental_B parsea DEPARTAMENTAL de derecha a izquierda usando los dominios válidos:
el sufijo siempre es una jurisdicción; lo que sobra son los valores desplazados. Solo rellena
columnas vacías (no sobreescribe) y marca en REVISAR_MANUAL lo que no se puede resolver sin adivinar.

In [5]:
na_antes = {c: int(B_raw[c].isna().sum()) for c in ["STATUS","MODALIDAD","JORNADA","PLAN"]}
dep_unicos_antes = B_raw["DEPARTAMENTAL"].nunique()

B_rep = L.reparar_departamental_B(B_raw)

na_despues = {c: int(B_rep[c].isna().sum()) for c in ["STATUS","MODALIDAD","JORNADA","PLAN"]}
recuperados = {c: na_antes[c]-na_despues[c] for c in na_antes}
print("Faltantes antes :", na_antes)
print("Faltantes después:", na_despues)
print("Recuperados      :", recuperados, "-> total", sum(recuperados.values()))
print("DEPARTAMENTAL únicos:", dep_unicos_antes, "->", B_rep["DEPARTAMENTAL"].nunique())
print("Marcados REVISAR_MANUAL:", int(B_rep["REVISAR_MANUAL"].notna().sum()))

log("DEPARTAMENTAL / STATUS·MODALIDAD·JORNADA·PLAN",
    "Desplazamiento de columnas: valores de 4 variables incrustados en DEPARTAMENTAL (817 registros).",
    "Parseo por dominios de derecha a izquierda; se devolvió cada valor a su columna y se dejó la jurisdicción en DEPARTAMENTAL.",
    sum(recuperados.values()),
    "La verificación mostró que el 100% de esos faltantes provenían del desplazamiento, no de pérdida real; son recuperables sin inventar datos.")

Faltantes antes : {'STATUS': 55, 'MODALIDAD': 55, 'JORNADA': 638, 'PLAN': 817}
Faltantes después: {'STATUS': 0, 'MODALIDAD': 0, 'JORNADA': 0, 'PLAN': 2}
Recuperados      : {'STATUS': 55, 'MODALIDAD': 55, 'JORNADA': 638, 'PLAN': 815} -> total 1563
DEPARTAMENTAL únicos: 120 -> 14
Marcados REVISAR_MANUAL: 4
[LOG] DEPARTAMENTAL / STATUS·MODALIDAD·JORNADA·PLAN: Parseo por dominios de derecha a izquierda; se devolvió cada valor a su columna y se dejó la jurisdicción en DEPARTAMENTAL. (afectados=1563)


In [6]:
# Casos que requieren revisión manual (no se resuelven automáticamente).
B_rep.loc[B_rep["REVISAR_MANUAL"].notna(), ["CODIGO","DEPARTAMENTAL","REVISAR_MANUAL"]]

,CODIGO,DEPARTAMENTAL,REVISAR_MANUAL
67,16-01-0982-46,ALTA VERAPAZ,texto no asignado tras reparar: 'SIN ESPECIFICAR'
1688,05-02-0065-46,ESCUINTLA,texto no asignado tras reparar: 'SIN ESPECIFICAR'
4168,13-01-0346-46,HUEHUETENANGO,texto no asignado tras reparar: 'INTERCALADO'
4641,13-27-0079-46,HUEHUETENANGO,texto no asignado tras reparar: 'INTERCALADO'


4. UNIFICACIÓN DE ESQUEMA Y UNIÓN EN UN SOLO CONJUNTO

- Grupo A: se elimina la columna vacía sin nombre y se fusionan SUPERVISOR + DIRECTOR.
- Grupo B: SUPERVISOR_DIRECTOR ya viene fusionado.
- Ambos se llevan al mismo conjunto de columnas y se concatenan.

In [7]:
# --- Grupo A: fusionar SUPERVISOR + DIRECTOR ---
A = A_raw.copy()

# Eliminar la columna sin nombre (100% vacía) del grupo A.
col_vacia = [c for c in A.columns if c.startswith("Unnamed") or c.strip() == ""]
A = A.drop(columns=col_vacia)
log("(columna sin nombre)", "Columna sin encabezado, 100% vacía (0 valores en 4033 registros).",
    f"Se eliminó la columna {col_vacia}.", len(A_raw),
    "No aporta información; su presencia rompía la unión con el Grupo B.", es_correccion=False)

# Convertir placeholder '----' de DIRECTOR a NA antes de fusionar.
n_placeholder = int((A["DIRECTOR"].astype(str).str.strip().str.fullmatch(r"-+")).sum())
A["SUPERVISOR"] = A["SUPERVISOR"].map(L.a_nulo_marcadores)
A["DIRECTOR"]   = A["DIRECTOR"].map(L.a_nulo_marcadores)

def fusionar(sup, dire):
    partes = [x for x in [sup, dire] if pd.notna(x) and str(x).strip() != ""]
    return " ".join(partes) if partes else pd.NA

A["SUPERVISOR_DIRECTOR"] = [fusionar(s, d) for s, d in zip(A["SUPERVISOR"], A["DIRECTOR"])]
A = A.drop(columns=["SUPERVISOR","DIRECTOR"])
A["REVISAR_MANUAL"] = pd.NA
log("DIRECTOR", "Marcador '----' usado como valor ausente (3 registros).",
    "Se convirtió '----' a NA.", n_placeholder,
    "'----' no es un nombre real; dejarlo falsearía los conteos de faltantes.")
log("SUPERVISOR + DIRECTOR", "En el Grupo B vienen fusionados; en A separados (estructuras distintas).",
    "Se fusionaron SUPERVISOR y DIRECTOR del Grupo A en SUPERVISOR_DIRECTOR para uniformar el esquema.",
    len(A), "Permite un único conjunto con estructura consistente en los 22 departamentos.", es_correccion=False)

[LOG] (columna sin nombre): Se eliminó la columna ['Unnamed: 1']. (afectados=4033)
[LOG] DIRECTOR: Se convirtió '----' a NA. (afectados=52)
[LOG] SUPERVISOR + DIRECTOR: Se fusionaron SUPERVISOR y DIRECTOR del Grupo A en SUPERVISOR_DIRECTOR para uniformar el esquema. (afectados=4033)


In [8]:
# --- Marcar origen y unir ---
A["GRUPO_ORIGEN"] = "A"
B_rep["GRUPO_ORIGEN"] = "B"

comunes = ["CODIGO","DISTRITO","DEPARTAMENTO","MUNICIPIO","ESTABLECIMIENTO","DIRECCION",
           "TELEFONO","SUPERVISOR_DIRECTOR","NIVEL","SECTOR","AREA","STATUS","MODALIDAD",
           "JORNADA","PLAN","DEPARTAMENTAL","GRUPO_ORIGEN","REVISAR_MANUAL"]

df = pd.concat([A[comunes], B_rep[comunes]], ignore_index=True)
print("Conjunto unido:", df.shape)
assert len(df) == len(A_raw) + len(B_raw), "La unión no conserva el total de registros"
df.head(3)

Conjunto unido: (9706, 18)


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR_DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,GRUPO_ORIGEN,REVISAR_MANUAL
0,17-01-0035-46,17-001,PETEN,FLORES,INSTITUTO PRIVADO DE EDUCACION DIVERSIFICADA D...,BARRIO SANTA ELENA,NaN,LILIN MELISA HERRERA ORREGO,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETÉN,A,<NA>
1,17-01-0038-46,NaN,PETEN,FLORES,INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFI...,BARRIO SANTA ELENA,NaN,NaN,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETÉN,A,<NA>
2,17-01-0039-46,17-,PETEN,FLORES,INSTITUTO NOCTURNO DE CIENCIAS COMERCIALES Y A...,BARRIO SANTA ELENA,NaN,JOSE FRANCISCO OLAN PONCE,DIVERSIFICADO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETÉN,A,<NA>


5. LIMPIEZA POR VARIABLE

5.1 MARCADORES DE AUSENCIA → NA
Convertimos '', -, ., N/A, NULL, SIN DATO, etc. a NA real en todas las columnas de texto.

In [9]:
cols_texto = ["DISTRITO","DEPARTAMENTO","MUNICIPIO","ESTABLECIMIENTO","DIRECCION","TELEFONO",
              "SUPERVISOR_DIRECTOR","SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","DEPARTAMENTAL"]
antes_na = int(df[cols_texto].isna().sum().sum())
for c in cols_texto:
    df[c] = df[c].map(L.a_nulo_marcadores)
nuevos = int(df[cols_texto].isna().sum().sum()) - antes_na
log("(todas las de texto)", "Marcadores de dato ausente ('-', '.', 'N/A', 'NULL', 'SIN DATO', etc.).",
    "Se normalizaron a NA con L.a_nulo_marcadores.", nuevos,
    "Unifica el criterio de 'faltante' para el análisis posterior.")

[LOG] (todas las de texto): Se normalizaron a NA con L.a_nulo_marcadores. (afectados=4)


5.2 NORMALIZACIÓN DE TEXTO
Caracteres invisibles (BOM, zero-width, \xa0), espacios al inicio/final y espacios múltiples.

In [10]:
afectados_norm = 0
for c in cols_texto + ["SUPERVISOR_DIRECTOR"]:
    original = df[c].copy()
    df[c] = df[c].map(L.normalizar_texto)
    afectados_norm += int((original.fillna("") != df[c].fillna("")).sum())
log("(todas las de texto)", "Espacios iniciales/finales, espacios múltiples y caracteres invisibles.",
    "Trim + colapso de espacios + eliminación de BOM/zero-width con L.normalizar_texto.",
    afectados_norm, "La actividad 5c y la validación exigen texto sin espacios sobrantes ni invisibles.")

[LOG] (todas las de texto): Trim + colapso de espacios + eliminación de BOM/zero-width con L.normalizar_texto. (afectados=14)


5.3 TILDES Y MAYÚSCULAS CONSISTENTES
Criterio unificado: categóricas y nombres geográficos en MAYÚSCULAS y sin tildes (así ya venían
DEPARTAMENTO y la mayoría; unifica el desajuste DEPARTAMENTO vs DEPARTAMENTAL y corrige la
tilde invertida 'EDUCACIÒN').

In [11]:
cols_sin_tilde = ["DEPARTAMENTO","MUNICIPIO","SECTOR","AREA","STATUS","MODALIDAD","JORNADA",
                  "PLAN","DEPARTAMENTAL","NIVEL"]
afectados_tilde = 0
for c in cols_sin_tilde:
    original = df[c].copy()
    df[c] = df[c].map(lambda x: L.quitar_tildes(x).upper() if pd.notna(x) else x)
    afectados_tilde += int((original.fillna("") != df[c].fillna("")).sum())

# ESTABLECIMIENTO, DIRECCION y nombres: solo mayúsculas + corregir tilde invertida, conservando ñ.
for c in ["ESTABLECIMIENTO","DIRECCION","SUPERVISOR_DIRECTOR"]:
    df[c] = df[c].map(lambda x: x.upper() if pd.notna(x) else x)
    # corregir acento grave usado por error en vocales (Ò->Ó, È->É, etc.)
    df[c] = df[c].map(lambda x: x.translate(str.maketrans("ÀÈÌÒÙ","ÁÉÍÓÚ")) if pd.notna(x) else x)

log("DEPARTAMENTO, DEPARTAMENTAL, categóricas",
    "Acentuación inconsistente (DEPARTAMENTO sin tildes vs DEPARTAMENTAL con tildes) y tilde invertida.",
    "Se pasó a MAYÚSCULAS y se quitaron tildes en categóricas/geográficas; se corrigió el acento grave en nombres.",
    afectados_tilde, "Evita que el mismo valor cuente como categorías distintas por diferencias de escritura.")

[LOG] DEPARTAMENTO, DEPARTAMENTAL, categóricas: Se pasó a MAYÚSCULAS y se quitaron tildes en categóricas/geográficas; se corrigió el acento grave en nombres. (afectados=2297)


5.4 TELÉFONOS
Se separan múltiples números en TELEFONO_1/TELEFONO_2, se valida el estándar de 8 dígitos y se
guarda N_TELEFONOS. La columna TELEFONO cruda se conserva como respaldo.

In [12]:
tel = df["TELEFONO"].map(L.separar_telefonos)
df["TELEFONO_1"] = [t[0] for t in tel]
df["TELEFONO_2"] = [t[1] for t in tel]
df["N_TELEFONOS"] = [t[2] for t in tel]
con_multiples = int((df["N_TELEFONOS"] >= 2).sum())
invalidos = int(((df["TELEFONO"].notna()) & (df["N_TELEFONOS"] == 0)).sum())
print("Registros con 2 teléfonos:", con_multiples, "| con teléfono presente pero inválido:", invalidos)
log("TELEFONO", "Celdas con más de un número, separadores distintos y números de != 8 dígitos.",
    "Se separó en TELEFONO_1/TELEFONO_2 (derivadas), validando 8 dígitos; N_TELEFONOS cuenta los válidos.",
    con_multiples + invalidos,
    "Un teléfono por columna permite validar formato; los de != 8 dígitos no cumplen el estándar de Guatemala.")

Registros con 2 teléfonos: 86 | con teléfono presente pero inválido: 56
[LOG] TELEFONO: Se separó en TELEFONO_1/TELEFONO_2 (derivadas), validando 8 dígitos; N_TELEFONOS cuenta los válidos. (afectados=142)


5.5 CÓDIGO DE DISTRITO
NN-NN-NNNN y NN-NNN son formatos válidos que se conservan; los truncados (17-, 10-) → NA.

In [13]:
dist = df["DISTRITO"].map(L.estandarizar_distrito)
invalidos_dist = int(sum(1 for _, ok in dist if not ok))
df["DISTRITO"] = [v for v, _ in dist]
log("DISTRITO", "Códigos truncados imposibles ('17-', '10-').",
    "Los códigos que no cumplen NN-NN-NNNN ni NN-NNN se convirtieron a NA.", invalidos_dist,
    "Un código truncado no identifica un distrito real; conservarlo induciría error.")

[LOG] DISTRITO: Los códigos que no cumplen NN-NN-NNNN ni NN-NNN se convirtieron a NA. (afectados=3)


5.6 NIVEL CONSTANTE
Todos los registros son DIVERSIFICADO (así se extrajo la fuente). Se conserva como constante
documentada en el libro de códigos (no se elimina para preservar la trazabilidad del filtro aplicado).

In [14]:
print("NIVEL:", df["NIVEL"].value_counts(dropna=False).to_dict())
log("NIVEL", "Variable constante (único valor DIVERSIFICADO).",
    "Se conserva como constante documentada (no se elimina).", len(df),
    "Documenta que el conjunto corresponde solo a nivel diversificado, según el filtro de extracción.",
    es_correccion=False)

NIVEL: {'DIVERSIFICADO': 9706}
[LOG] NIVEL: Se conserva como constante documentada (no se elimina). (afectados=9706)


5.7 CONSISTENCIA DE CATEGORÍAS (REVISIÓN CON SIMILITUD)
Buscamos categorías que signifiquen lo mismo pero escritas distinto dentro de cada variable
categórica. Tras la normalización de texto/tildes, listamos los dominios finales para justificar
cualquier unificación.

In [15]:
for c in ["SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","DEPARTAMENTO","DEPARTAMENTAL"]:
    print(f"--- {c} ({df[c].nunique(dropna=True)} categorías) ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")

--- SECTOR (4 categorías) ---
SECTOR
PRIVADO        7877
OFICIAL        1358
COOPERATIVA     295
MUNICIPAL       176 

--- AREA (2 categorías) ---
AREA
URBANA    7313
RURAL     2391
NaN          2 

--- STATUS (5 categorías) ---
STATUS
ABIERTA                    6038
CERRADA TEMPORALMENTE      2291
CERRADA DEFINITIVAMENTE    1286
TEMPORAL NOMBRAMIENTO        47
TEMPORAL TITULOS             44 

--- MODALIDAD (2 categorías) ---
MODALIDAD
MONOLINGUE    9239
BILINGUE       467 

--- JORNADA (6 categorías) ---
JORNADA
DOBLE          3217
VESPERTINA     3092
MATUTINA       2084
SIN JORNADA     952
NOCTURNA        296
INTERMEDIA       65 

--- PLAN (12 categorías) ---
PLAN
DIARIO(REGULAR)                          6038
FIN DE SEMANA                            2399
SEMIPRESENCIAL (FIN DE SEMANA)            428
SEMIPRESENCIAL (UN DIA A LA SEMANA)       426
A DISTANCIA                               148
SEMIPRESENCIAL                            100
SEMIPRESENCIAL (DOS DIAS A LA SEMANA)      58
VI

6. DUPLICADOS EXACTOS Y PARCIALES

La guía pide no eliminar automáticamente: se analizan y se documenta la decisión.

In [16]:
# --- Exactos ---
dup_exactos = int(df.drop(columns=["GRUPO_ORIGEN","REVISAR_MANUAL"]).duplicated().sum())
dup_codigo = int(df["CODIGO"].duplicated().sum())
print("Duplicados exactos:", dup_exactos, "| CODIGO repetidos:", dup_codigo)

Duplicados exactos: 0 | CODIGO repetidos: 0


Duplicados parciales (similitud de cadenas con RapidFuzz).
Definición estricta para evitar falsos positivos: dos registros del mismo DEPARTAMENTO y
MUNICIPIO cuyo ESTABLECIMIENTO es casi idéntico (token_sort_ratio ≥ 90) y comparten
dirección (≥ 90) o teléfono, y además coinciden en JORNADA, PLAN, SECTOR y AREA.
Es decir, filas que solo se diferencian por el CODIGO. Una definición laxa (solo nombre)
marca ~6000 pares que en realidad son la misma sede en jornadas/planes distintos, que sí son
registros legítimos.

In [17]:
def _norm(x):
    return re.sub(r"\s+", " ", str(x)).strip().upper() if pd.notna(x) else ""

posibles = []
df_idx = df.reset_index(drop=True)
for (dep, mun), g in df_idx.groupby(["DEPARTAMENTO","MUNICIPIO"], dropna=False, observed=True):
    idxs = g.index.tolist()
    nom  = g["ESTABLECIMIENTO"].map(_norm).tolist()
    dire = g["DIRECCION"].map(_norm).tolist()
    tel  = g["TELEFONO_1"].fillna("").tolist()
    key  = g[["JORNADA","PLAN","SECTOR","AREA"]].astype("object").fillna("").astype(str).agg("|".join, axis=1).tolist()
    for i in range(len(idxs)):
        for j in range(i+1, len(idxs)):
            if not nom[i] or not nom[j]:
                continue
            s_nom = fuzz.token_sort_ratio(nom[i], nom[j])
            if s_nom < 90:
                continue
            s_dir = fuzz.token_sort_ratio(dire[i], dire[j]) if dire[i] and dire[j] else 0
            mismo_tel = bool(tel[i]) and tel[i] == tel[j]
            if (s_dir >= 90 or mismo_tel) and key[i] == key[j]:
                posibles.append({
                    "cod_a": df_idx.at[idxs[i],"CODIGO"], "cod_b": df_idx.at[idxs[j],"CODIGO"],
                    "sim_nombre": s_nom, "sim_direccion": s_dir, "mismo_telefono": mismo_tel,
                    "nombre_a": nom[i], "nombre_b": nom[j], "depto": dep, "municipio": mun,
                    "idx_a": idxs[i], "idx_b": idxs[j]})
posibles = (pd.DataFrame(posibles).sort_values(["sim_nombre","sim_direccion"], ascending=False)
            if posibles else pd.DataFrame())
n_posibles_pares = len(posibles)
print("Pares de posibles duplicados parciales:", n_posibles_pares)
posibles.head(15)

Pares de posibles duplicados parciales: 1030


,cod_a,cod_b,sim_nombre,sim_direccion,mismo_telefono,nombre_a,nombre_b,depto,municipio,idx_a,idx_b
0,16-13-0225-46,16-13-0226-46,100.0,100.0,False,LICEO MIXTO EN COMPUTACION DEL NORTE 1A.,LICEO MIXTO EN COMPUTACION DEL NORTE 1A.,ALTA VERAPAZ,CHISEC,4411,4412
2,16-01-0559-46,16-01-8849-46,100.0,100.0,False,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,ALTA VERAPAZ,COBAN,4051,4154
3,16-01-0565-46,16-01-0646-46,100.0,100.0,False,INSTITUTO MIXTO PRIVADO SALUD Y DESARROLLO DIA...,INSTITUTO MIXTO PRIVADO SALUD Y DESARROLLO DIA...,ALTA VERAPAZ,COBAN,4053,4070
4,16-01-0587-46,16-01-0588-46,100.0,100.0,True,LICEO MIXTO EN COMPUTACION DEL NORTE 1A.,LICEO MIXTO EN COMPUTACION DEL NORTE 1A.,ALTA VERAPAZ,COBAN,4060,4061
6,16-16-0135-46,16-16-0139-46,100.0,100.0,True,LICEO CRISTIANO EBENEZER,LICEO CRISTIANO EBENEZER,ALTA VERAPAZ,LA TINTA,4488,4491
10,16-03-0088-46,16-03-1520-46,100.0,100.0,True,LICEO GALILEO GALILEI 2A.,LICEO GALILEO GALILEI 2A.,ALTA VERAPAZ,SAN CRISTOBAL VERAPAZ,4178,4210
11,16-03-0271-46,16-03-0272-46,100.0,100.0,True,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,ALTA VERAPAZ,SAN CRISTOBAL VERAPAZ,4201,4202
12,16-03-0271-46,16-03-0273-46,100.0,100.0,True,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,ALTA VERAPAZ,SAN CRISTOBAL VERAPAZ,4201,4203
13,16-03-0272-46,16-03-0273-46,100.0,100.0,True,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,COLEGIO DE FORMACIÓN TÉCNICO-PROFESIONAL NEXUS 0,ALTA VERAPAZ,SAN CRISTOBAL VERAPAZ,4202,4203
14,16-03-9884-46,16-03-9885-46,100.0,100.0,True,LICEO SAN CRISTOBAL 5A.,LICEO SAN CRISTOBAL 5A.,ALTA VERAPAZ,SAN CRISTOBAL VERAPAZ,4212,4213


In [18]:
# Decisión documentada (la guía prohíbe borrado automático): NO se eliminan.
# CODIGO es la llave única oficial del MINEDUC; cada registro es una inscripción
# administrativa distinta aunque coincidan nombre/dirección/teléfono/jornada. Se conservan
# y se marcan en una columna propia (ES_POSIBLE_DUPLICADO) para revisión, sin tocar
# REVISAR_MANUAL, que queda reservado a los 4 casos de parseo del Grupo B.
df["ES_POSIBLE_DUPLICADO"] = False
if n_posibles_pares:
    idxs_dup = pd.unique(posibles[["idx_a","idx_b"]].values.ravel())
    df.loc[idxs_dup, "ES_POSIBLE_DUPLICADO"] = True
n_filas_dup = int(df["ES_POSIBLE_DUPLICADO"].sum())
print("Filas marcadas como posible duplicado:", n_filas_dup)
log("(registro completo)",
    f"{dup_exactos} duplicados exactos; {n_posibles_pares} pares ({n_filas_dup} filas) de posibles duplicados parciales.",
    "Exactos: no hay. Parciales: se conservan y se marcan en ES_POSIBLE_DUPLICADO; se exportan a posibles_duplicados.csv.",
    n_filas_dup,
    "CODIGO es llave única oficial; son inscripciones distintas, no se eliminan (la guía exige análisis caso por caso).",
    es_correccion=False)

Filas marcadas como posible duplicado: 1724
[LOG] (registro completo): Exactos: no hay. Parciales: se conservan y se marcan en ES_POSIBLE_DUPLICADO; se exportan a posibles_duplicados.csv. (afectados=1724)


7. CONSISTENCIA ENTRE VARIABLES
Verificamos que DEPARTAMENTAL sea coherente con DEPARTAMENTO (la jurisdicción debe pertenecer al
mismo departamento) y documentamos las subdivisiones administrativas legítimas (Guatemala y Quiché).

In [19]:
def depto_de_jurisdiccion(j):
    if pd.isna(j): return j
    for d in L.DEPARTAMENTOS:
        if j == d or j.startswith(d + " "):
            return d
    return "?"
chk = df.copy()
chk["DEPTO_DE_DEPARTAMENTAL"] = chk["DEPARTAMENTAL"].map(depto_de_jurisdiccion)
incons = chk[(chk["DEPARTAMENTAL"].notna()) & (chk["DEPTO_DE_DEPARTAMENTAL"] != chk["DEPARTAMENTO"]) &
             (chk["DEPTO_DE_DEPARTAMENTAL"] != "?")]
print("Inconsistencias DEPARTAMENTO vs DEPARTAMENTAL:", len(incons))
print("Subdivisiones detectadas (legítimas):")
print(df.loc[df["DEPARTAMENTAL"].str.contains(" ", na=False), ["DEPARTAMENTO","DEPARTAMENTAL"]]
      .drop_duplicates().sort_values(["DEPARTAMENTO","DEPARTAMENTAL"]).to_string(index=False))

Inconsistencias DEPARTAMENTO vs DEPARTAMENTAL: 0
Subdivisiones detectadas (legítimas):
DEPARTAMENTO       DEPARTAMENTAL
ALTA VERAPAZ        ALTA VERAPAZ
BAJA VERAPAZ        BAJA VERAPAZ
 EL PROGRESO         EL PROGRESO
   GUATEMALA     GUATEMALA NORTE
   GUATEMALA GUATEMALA OCCIDENTE
   GUATEMALA   GUATEMALA ORIENTE
   GUATEMALA       GUATEMALA SUR
      QUICHE        QUICHE NORTE
  SAN MARCOS          SAN MARCOS
  SANTA ROSA          SANTA ROSA


8. ORDEN FINAL DE COLUMNAS Y TIPOS

In [20]:
df["NIVEL"] = df["NIVEL"].astype("category")
for c in ["DEPARTAMENTO","MUNICIPIO","SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN",
          "DEPARTAMENTAL","GRUPO_ORIGEN"]:
    df[c] = df[c].astype("category")
df["N_TELEFONOS"] = df["N_TELEFONOS"].astype("int64")

orden = ["CODIGO","DISTRITO","DEPARTAMENTO","MUNICIPIO","ESTABLECIMIENTO","DIRECCION",
         "TELEFONO","TELEFONO_1","TELEFONO_2","N_TELEFONOS","SUPERVISOR_DIRECTOR","NIVEL",
         "SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","DEPARTAMENTAL",
         "GRUPO_ORIGEN","ES_POSIBLE_DUPLICADO","REVISAR_MANUAL"]
df = df[orden]
df.dtypes

CODIGO                       str
DISTRITO                     str
DEPARTAMENTO            category
MUNICIPIO               category
ESTABLECIMIENTO              str
DIRECCION                    str
TELEFONO                     str
TELEFONO_1                   str
TELEFONO_2                   str
N_TELEFONOS                int64
SUPERVISOR_DIRECTOR          str
NIVEL                   category
SECTOR                  category
AREA                    category
STATUS                  category
MODALIDAD               category
JORNADA                 category
PLAN                    category
DEPARTAMENTAL           category
GRUPO_ORIGEN            category
ES_POSIBLE_DUPLICADO        bool
REVISAR_MANUAL            object
dtype: object

9. VALIDACIÓN DEL CONJUNTO LIMPIO (PRUEBAS AUTOMÁTICAS)
Actividad 7: cada assert verifica una regla de calidad. Si alguno falla, el notebook se detiene.

In [21]:
def check(nombre, condicion):
    estado = "OK" if condicion else "FALLA"
    print(f"[{estado}] {nombre}")
    assert condicion, f"Falló la validación: {nombre}"

cols_finales = [c for c in df.columns if c not in ["GRUPO_ORIGEN","REVISAR_MANUAL"]]
check("No hay duplicados exactos", df[cols_finales].duplicated().sum() == 0)
check("CODIGO es único", df["CODIGO"].is_unique)
check("Sin espacios al inicio/final en texto",
      all(df[c].dropna().map(lambda x: x == x.strip()).all()
          for c in ["ESTABLECIMIENTO","DIRECCION","SUPERVISOR_DIRECTOR","MUNICIPIO"]))
check("Teléfonos válidos tienen 8 dígitos",
      df["TELEFONO_1"].dropna().map(lambda x: bool(re.fullmatch(r"\d{8}", x))).all() and
      df["TELEFONO_2"].dropna().map(lambda x: bool(re.fullmatch(r"\d{8}", x))).all())
check("DEPARTAMENTO en el catálogo de 22", set(df["DEPARTAMENTO"].dropna()).issubset(set(L.DEPARTAMENTOS)))
check("DISTRITO con formato válido o NA",
      df["DISTRITO"].dropna().map(lambda x: bool(re.fullmatch(r"\d{2}-\d{2}-\d{4}|\d{2}-\d{3}", x))).all())
check("STATUS en dominio",
      set(df["STATUS"].dropna().astype(str)).issubset(set(s.upper() for s in L.STATUS_DOM)))
check("NIVEL constante DIVERSIFICADO", set(df["NIVEL"].dropna().astype(str)) == {"DIVERSIFICADO"})
print("\nTodas las pruebas de calidad pasaron.")

[OK] No hay duplicados exactos
[OK] CODIGO es único
[OK] Sin espacios al inicio/final en texto
[OK] Teléfonos válidos tienen 8 dígitos
[OK] DEPARTAMENTO en el catálogo de 22
[OK] DISTRITO con formato válido o NA
[OK] STATUS en dominio
[OK] NIVEL constante DIVERSIFICADO

Todas las pruebas de calidad pasaron.


10. INFORME DE CALIDAD — ANTES VS. DESPUÉS (ACTIVIDAD 8)

In [22]:
na_desp = int(df[cols_finales].isna().sum().sum())
celdas = len(df) * len(cols_finales)
errores_corregidos = sum(r["Registros afectados"] for r in REGISTRO
                         if r.get("_es_correccion") and isinstance(r["Registros afectados"], int))
DESPUES = {
    "Registros": len(df),
    "Variables": df.shape[1],
    "Valores faltantes": f"{na_desp} ({round(100*na_desp/celdas,2)}%)",
    "Variables con NA": int((df[cols_finales].isna().sum() > 0).sum()),
    "Duplicados exactos": int(df[cols_finales].duplicated().sum()),
    "Posibles duplicados": f"{n_filas_dup} filas ({n_posibles_pares} pares), conservadas y marcadas",
    "Variables con formato inconsistente": "0 (corregidas: DISTRITO, TELEFONO, tildes)",
    "Variables con tipo incorrecto": "0 (categóricas y numéricas tipadas)",
    "Categorías inconsistentes": "0 (unificadas por tildes/mayúsculas)",
    "Errores corregidos": errores_corregidos,
}
informe = pd.DataFrame({
    "Métrica": list(DESPUES.keys()),
    "Antes": [ANTES.get(k, "—") for k in DESPUES],
    "Después": list(DESPUES.values()),
})
informe

,Métrica,Antes,Después
0,Registros,9706,9706
1,Variables,19,22
2,Valores faltantes,29446 (15.97%),12686 (6.54%)
3,Variables con NA,13,9
4,Duplicados exactos,0,0
5,Posibles duplicados,—,"1724 filas (1030 pares), conservadas y marcadas"
6,Variables con formato inconsistente,—,"0 (corregidas: DISTRITO, TELEFONO, tildes)"
7,Variables con tipo incorrecto,—,0 (categóricas y numéricas tipadas)
8,Categorías inconsistentes,—,0 (unificadas por tildes/mayúsculas)
9,Errores corregidos,—,4075


11. REGISTRO DE TRANSFORMACIONES (ACTIVIDAD 6)

In [23]:
registro_df = pd.DataFrame(REGISTRO)[["Variable","Problema detectado","Transformación",
                                       "Registros afectados","Justificación"]]
registro_df

,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,DEPARTAMENTAL / STATUS·MODALIDAD·JORNADA·PLAN,Desplazamiento de columnas: valores de 4 varia...,Parseo por dominios de derecha a izquierda; se...,1563,La verificación mostró que el 100% de esos fal...
1,(columna sin nombre),"Columna sin encabezado, 100% vacía (0 valores ...",Se eliminó la columna ['Unnamed: 1'].,4033,No aporta información; su presencia rompía la ...
2,DIRECTOR,Marcador '----' usado como valor ausente (3 re...,Se convirtió '----' a NA.,52,'----' no es un nombre real; dejarlo falsearía...
3,SUPERVISOR + DIRECTOR,En el Grupo B vienen fusionados; en A separado...,Se fusionaron SUPERVISOR y DIRECTOR del Grupo ...,4033,Permite un único conjunto con estructura consi...
4,(todas las de texto),"Marcadores de dato ausente ('-', '.', 'N/A', '...",Se normalizaron a NA con L.a_nulo_marcadores.,4,Unifica el criterio de 'faltante' para el anál...
5,(todas las de texto),"Espacios iniciales/finales, espacios múltiples...",Trim + colapso de espacios + eliminación de BO...,14,La actividad 5c y la validación exigen texto s...
6,"DEPARTAMENTO, DEPARTAMENTAL, categóricas",Acentuación inconsistente (DEPARTAMENTO sin ti...,Se pasó a MAYÚSCULAS y se quitaron tildes en c...,2297,Evita que el mismo valor cuente como categoría...
7,TELEFONO,"Celdas con más de un número, separadores disti...",Se separó en TELEFONO_1/TELEFONO_2 (derivadas)...,142,Un teléfono por columna permite validar format...
8,DISTRITO,"Códigos truncados imposibles ('17-', '10-').",Los códigos que no cumplen NN-NN-NNNN ni NN-NN...,3,Un código truncado no identifica un distrito r...
9,NIVEL,Variable constante (único valor DIVERSIFICADO).,Se conserva como constante documentada (no se ...,9706,Documenta que el conjunto corresponde solo a n...


12. GENERACIÓN DEL CONJUNTO LIMPIO ÚNICO (ACTIVIDAD 9)

In [24]:
os.makedirs("Data/limpio", exist_ok=True)
salida = "Data/limpio/establecimientos_diversificado_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

registro_df.to_csv("Data/limpio/registro_transformaciones.csv", index=False, encoding="utf-8-sig")
informe.to_csv("Data/limpio/informe_calidad.csv", index=False, encoding="utf-8-sig")
if n_posibles_pares:
    posibles.to_csv("Data/limpio/posibles_duplicados.csv", index=False, encoding="utf-8-sig")

print("Conjunto limpio guardado en:", salida)
print("Dimensiones finales:", df.shape)
print("Departamentos:", df['DEPARTAMENTO'].nunique(), "| Registros:", len(df))
df.head(5)

Conjunto limpio guardado en: Data/limpio/establecimientos_diversificado_limpio.csv
Dimensiones finales: (9706, 22)
Departamentos: 22 | Registros: 9706


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,TELEFONO_1,TELEFONO_2,N_TELEFONOS,SUPERVISOR_DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,GRUPO_ORIGEN,ES_POSIBLE_DUPLICADO,REVISAR_MANUAL
0,17-01-0035-46,17-001,PETEN,FLORES,INSTITUTO PRIVADO DE EDUCACION DIVERSIFICADA D...,BARRIO SANTA ELENA,NaN,NaN,NaN,0,LILIN MELISA HERRERA ORREGO,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETEN,A,False,<NA>
1,17-01-0038-46,NaN,PETEN,FLORES,INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFI...,BARRIO SANTA ELENA,NaN,NaN,NaN,0,NaN,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETEN,A,False,<NA>
2,17-01-0039-46,NaN,PETEN,FLORES,INSTITUTO NOCTURNO DE CIENCIAS COMERCIALES Y A...,BARRIO SANTA ELENA,NaN,NaN,NaN,0,JOSE FRANCISCO OLAN PONCE,DIVERSIFICADO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),PETEN,A,False,<NA>
3,17-01-0058-46,17-001,PETEN,FLORES,"LICEO PETENERO EN CIENCIAS ADMINISTRATIVAS, RE...",SANTA ELENA DE LA CRUZ,NaN,NaN,NaN,0,LILIN MELISA HERRERA ORREGO ERWIN AUGUSTO LÓPE...,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,DOBLE,FIN DE SEMANA,PETEN,A,False,<NA>
4,17-01-0064-46,17-01-1026,PETEN,FLORES,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,ALDEA IXLU,54823795,54823795,NaN,1,BRAULIO AMILCAR CHÁN TESUCÚN VICENTE AVELAR HE...,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),PETEN,A,False,<NA>
